In [1]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
if os.path.basename(os.getcwd()) == 'notes': os.chdir('..')
print('工作目录:', os.getcwd())

工作目录: d:\github\prediction


In [ ]:
# ============================================================
# 从 NetCDF 读取 #0 站点 (1141A 普陀) → DataFrame → 80/15/5 分割 → 预处理
# 先分割再插值, 杜绝边界数据泄漏
# ============================================================
import netCDF4 as nc

def load_station(nc_path, idx=0):
    """从 NetCDF 读取指定站点, 转为带 datetime 索引的 DataFrame"""
    f = nc.Dataset(nc_path)
    t = f.variables['time'][:]
    dt = pd.Timestamp(f.variables['time'].units.split('since ')[1]) + pd.to_timedelta(t, unit='h')
    # 变量映射: NetCDF → 旧列名
    pm25 = f.variables['PM2.5'][:, idx]
    temp = f.variables['t2m'][:, idx] - 273.15       # K → °C
    dewp = f.variables['d2m'][:, idx] - 273.15       # K → °C
    pres = f.variables['sp'][:, idx] / 100.0         # Pa → hPa
    tp   = f.variables['tp'][:, idx] * 1000.0        # m → mm
    u, v = f.variables['u100'][:, idx], f.variables['v100'][:, idx]
    iws  = np.sqrt(u**2 + v**2)                       # 风速 m/s
    # 风向分类: cv(静风)/NE/SE/SW/NW
    wdir = np.degrees(np.arctan2(-u, -v)) % 360
    cbwd = np.where(iws < 0.5, 'cv',
           np.where(wdir < 90, 'NE', np.where(wdir < 180, 'SE',
           np.where(wdir < 270, 'SW', 'NW')))).astype('<U2')
    # 湿度: Magnus 公式
    a, b = 17.625, 243.04
    humi = np.clip(100 * np.exp(a*dewp/(dewp+b+1e-10)) / np.exp(a*temp/(temp+b+1e-10)), 0, 100)
    f.close()
    df = pd.DataFrame({'PM_Jingan': pm25, 'PM_US Post': pm25, 'PM_Xuhui': pm25,
        'DEWP': dewp, 'HUMI': humi, 'PRES': pres, 'TEMP': temp,
        'cbwd': cbwd, 'Iws': iws, 'precipitation': tp, 'Iprec': np.cumsum(tp)}, index=dt)
    df.index.name = 'datetime'
    return df.resample('1h').first()  # 补齐完整 1h 时间轴

def split_and_preprocess(df, ratios=(0.80, 0.15, 0.05)):
    """按时间分割 → 各子集独立预处理 (无泄漏) → 返回 train/val/test"""
    n = len(df); n1 = int(n * ratios[0]); n2 = int(n * sum(ratios[:2]))
    subs = []
    for raw in [df.iloc[:n1], df.iloc[n1:n2], df.iloc[n2:]]:
        d = raw.copy()
        d['pm_ave'] = d[['PM_Jingan','PM_US Post','PM_Xuhui']].mean(axis=1)
        d['pm_ave'] = d['pm_ave'].interpolate(method='time', limit=168).ffill().bfill()
        subs.append(d)
    return subs  # df_tr, df_va, df_te

df_raw = load_station('data3/dataset_yrd.nc', idx=0)
df_tr, df_va, df_te = split_and_preprocess(df_raw)
print(f'训练 {len(df_tr)} | 验证 {len(df_va)} | 测试 {len(df_te)}')
print(f'pm_ave 范围: {df_tr.pm_ave.min():.1f} ~ {df_tr.pm_ave.max():.1f} ug/m3')

# 基线模型: 训练集每小时历史均值
hourly_baseline = df_tr.groupby(df_tr.index.hour)['pm_ave'].mean()

In [ ]:
# ============================================================
# 36 组评估: 每组 24h 上下文 + 48h 预测, RMSE (ug/m3)
# 基线1: 按"一天各小时"的训练集均值 (逐小时气候)
# 同时报告 48h RMSE 与前 24h RMSE (参考)
# ============================================================
N_GROUPS, N_CTX, N_PRED = 36, 24, 48
STRIDE = (len(df_te) - N_CTX - N_PRED) // (N_GROUPS - 1)
pm_te = df_te['pm_ave'].values

rmses, rmses24 = [], []
fig, axes = plt.subplots(6, 6, figsize=(36, 36)); axes = axes.flatten()
for g in range(N_GROUPS):
    s = g * STRIDE
    hours = df_te.index[s+N_CTX:s+N_CTX+N_PRED].hour
    preds = hours.map(hourly_baseline).values
    actual = pm_te[s+N_CTX:s+N_CTX+N_PRED]
    err2 = (preds - actual) ** 2
    rmses.append(np.sqrt(err2.mean()))          # 48h RMSE
    rmses24.append(np.sqrt(err2[:24].mean()))   # 前 24h RMSE (参考)
    ax = axes[g]; h = np.arange(1, N_PRED+1)
    ax.plot(h, actual, lw=0.8, label='actual')
    ax.plot(h, preds, lw=0.8, label='predicted')
    ax.set_title(f'G{g+1} RMSE={rmses[-1]:.1f}', fontsize=9)
    ax.legend(fontsize=7); ax.set_xlabel('hour'); ax.set_ylabel('PM2.5')
plt.suptitle(f'Average(hour-of-day) — {N_GROUPS} Groups (RMSE 48h)', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()
rmses = np.array(rmses); rmses24 = np.array(rmses24)
print(f'\n=== {N_GROUPS} 组 RMSE 汇总 (48h | 24h) ===')
for g in range(N_GROUPS): print(f'  G{g+1:2d}: {rmses[g]:6.2f} | {rmses24[g]:6.2f}')
print(f'\n平均 RMSE(48h) = {rmses.mean():.2f} | 参考 RMSE(24h) = {rmses24.mean():.2f}')

In [ ]:
# ============================================================
# 基线2: 常数预测 = 训练集"所有数据"的单一平均值 (最朴素基线)
# 每个未来时刻的预测值恒等于该均值; 报告 48h 与 24h RMSE
# ============================================================
global_mean = df_tr['pm_ave'].mean()
print(f'训练集全体均值 = {global_mean:.2f} ug/m3')

rmses, rmses24 = [], []
fig, axes = plt.subplots(6, 6, figsize=(36, 36)); axes = axes.flatten()
for g in range(N_GROUPS):
    s = g * STRIDE
    preds = np.full(N_PRED, global_mean)        # 常数预测
    actual = pm_te[s+N_CTX:s+N_CTX+N_PRED]
    err2 = (preds - actual) ** 2
    rmses.append(np.sqrt(err2.mean()))
    rmses24.append(np.sqrt(err2[:24].mean()))
    ax = axes[g]; h = np.arange(1, N_PRED+1)
    ax.plot(h, actual, lw=0.8, label='actual')
    ax.plot(h, preds, lw=0.8, label='predicted')
    ax.set_title(f'G{g+1} RMSE={rmses[-1]:.1f}', fontsize=9)
    ax.legend(fontsize=7); ax.set_xlabel('hour'); ax.set_ylabel('PM2.5')
plt.suptitle(f'Global-Mean Baseline — {N_GROUPS} Groups (RMSE 48h)', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()
rmses = np.array(rmses); rmses24 = np.array(rmses24)
print(f'\n=== 常数(均值)基线 {N_GROUPS} 组 (48h | 24h) ===')
for g in range(N_GROUPS): print(f'  G{g+1:2d}: {rmses[g]:6.2f} | {rmses24[g]:6.2f}')
print(f'\n平均 RMSE(48h) = {rmses.mean():.2f} | 参考 RMSE(24h) = {rmses24.mean():.2f}')

In [ ]:
# ============================================================
# 基线3: 持久性预测 (Persistence) — 未来48h常数等于"最近一小时"观测值
# 最经典的时间序列朴素基线: 明天跟今天一样; 报告 48h 与 24h RMSE
# ============================================================
print(f'持久性基线: 未来48h全部用最近一小时观测值常数外推\n')

rmses, rmses24 = [], []
fig, axes = plt.subplots(6, 6, figsize=(36, 36)); axes = axes.flatten()
for g in range(N_GROUPS):
    s = g * STRIDE
    last_known = pm_te[s + N_CTX - 1]              # 上下文最后一小时 (最近观测)
    preds = np.full(N_PRED, last_known)             # 常数预测
    actual = pm_te[s + N_CTX:s + N_CTX + N_PRED]
    err2 = (preds - actual) ** 2
    rmses.append(np.sqrt(err2.mean()))              # 48h RMSE
    rmses24.append(np.sqrt(err2[:24].mean()))       # 前 24h RMSE (参考)
    ax = axes[g]; h = np.arange(1, N_PRED + 1)
    ax.plot(h, actual, lw=0.8, label='actual')
    ax.plot(h, preds, lw=0.8, label='predicted')
    ax.set_title(f'G{g+1} RMSE={rmses[-1]:.1f}', fontsize=9)
    ax.legend(fontsize=7); ax.set_xlabel('hour'); ax.set_ylabel('PM2.5')
plt.suptitle(f'Persistence Baseline — {N_GROUPS} Groups (RMSE 48h)', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()
rmses = np.array(rmses); rmses24 = np.array(rmses24)
print(f'\n=== 持久性基线 {N_GROUPS} 组 (48h | 24h) ===')
for g in range(N_GROUPS): print(f'  G{g+1:2d}: {rmses[g]:6.2f} | {rmses24[g]:6.2f}')
print(f'\n平均 RMSE(48h) = {rmses.mean():.2f} | 参考 RMSE(24h) = {rmses24.mean():.2f}')